In [1]:
pip install rugbypy

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [3]:
from rugbypy.match import *
from rugbypy.competition import *
from rugbypy.team import *
from rugbypy.venue import *
from rugbypy.player import *

In [4]:
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

In [36]:
pd.options.mode.chained_assignment = None

In [37]:
## Reading in data

test_matches = pd.read_excel("detailed_test_matches.xlsx")
match_stats = pd.read_excel("aggregated_stats.xlsx")
test_nations = pd.read_excel("test_nations.xlsx")
team_stats = pd.read_excel("team_stats.xlsx")
venues = fetch_all_venues()

Fetching all venues...


In [38]:
## Only considering test matches between Tier 1 Nations

test_matches = test_matches[test_matches['home_team'].isin(test_nations['team_name']) &
                            test_matches['away_team'].isin(test_nations['team_name'])]

In [39]:
## Removal of Fiji - no national stadium where they're the home team

test_matches = test_matches[test_matches['home_team'] != "Fiji"]

In [40]:
test_match_results = test_matches[['match_id', 'date', 'season', 'competition_id', 'competition', 'venue_id', 'venue', 
                                    'home_team', 'home_team_id', 'away_team', 'away_team_id', 'home_score', 'away_score']]

test_match_results['winner'] = np.where(test_match_results['home_score'] > test_match_results['away_score'],
                                        test_match_results['home_team'], test_match_results['away_team'])

test_match_results['home_team_win'] = np.where(test_match_results['winner'] == test_match_results['home_team'], 1, 0)

test_match_results['neutral_match'] = np.where((test_match_results['competition'] == "Rugby World Cup") &
                                                (test_match_results['home_team'] != "France"),
                                                "neutral", "NA")

## test_match_results - all tests including neutral matches

home_away_tests = test_match_results[test_match_results['neutral_match'] != "neutral"]

home_away_tests

## home_away_tests - test matches where there is a clear home/away distinction. Ignores neutral-site 


,match_id,date,season,competition_id,competition,venue_id,venue,home_team,home_team_id,away_team,away_team_id,home_score,away_score,winner,home_team_win,neutral_match
0,26eb0d57,20220205,2022,7a1361e7,Six Nations,07b0586c,Aviva Stadium,Ireland,165b36e5,Wales,094dae60,29,7,Ireland,1,NA
1,d803469c,20220205,2022,7a1361e7,Six Nations,a35811c5,Scottish Gas Murrayfield,Scotland,4409215c,England,f60f757d,20,17,Scotland,1,NA
2,4a49c716,20220206,2022,7a1361e7,Six Nations,5b691aae,Stade de France,France,67d88578,Italy,d0cffe5e,37,10,France,1,NA
3,7ec3e5e5,20220212,2022,7a1361e7,Six Nations,fe933f9f,Principality Stadium,Wales,094dae60,Scotland,4409215c,20,17,Wales,1,NA
4,71ccf2fb,20220212,2022,7a1361e7,Six Nations,5b691aae,Stade de France,France,67d88578,Ireland,165b36e5,30,24,France,1,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341,337dd942,20260808,2026,96c775f2,International Test Match,b9cf87a3,Estadio José Amalfitani,Argentina,11392c0c,South Africa,9b93d911,10,17,South Africa,0,NA
342,c09a6180,20260815,2026,96c775f2,International Test Match,eb9c53f6,Queensland Country Bank Stadium,Australia,b904fef9,Japan,f97333f3,56,17,Australia,1,NA
343,a215a140,20260822,2026,96c775f2,International Test Match,e952f544,Ellis Park,South Africa,9b93d911,New Zealand,d75e45cc,16,33,New Zealand,0,NA
344,e1440199,20260829,2026,96c775f2,International Test Match,3ce3079c,DHL Stadium,South Africa,9b93d911,New Zealand,d75e45cc,33,26,South Africa,1,NA


In [41]:
## New Zealand v. South Africa at Twickenham (neutral-site RWC Preparation Match)

home_away_tests = home_away_tests[~home_away_tests['match_id'].isin(['db7135be'])]

In [42]:
test_venues = venues[venues['venue_id'].isin(test_match_results['venue_id'])]

test_venues = test_venues[~test_venues['venue_name'].isin(['Wembley Stadium', 'Soldier Field'])]

In [43]:
match_stats = match_stats[match_stats['team'].isin(test_nations['team_name'])]

In [44]:
## Data validation

match_stats = match_stats.rename(columns = {"team": "team_name"})

match_stats = match_stats[match_stats['turnovers_lost'] < 30]


In [45]:
match_stats = pd.merge(test_nations, match_stats, how = 'inner', on = 'team_name')

match_stats


,team_id,team_name,match_id,carries,line_breaks,tackles_completed,turnovers_lost,turnovers_won,kicks,passes,...,offload,penalty_goals,points,rucks_won,runs,tackles,total_free_kicks_conceded,total_lineouts,turnover_knock_on,turnovers_conceded
0,67d88578,France,3693cfdf,87,6,0,0,0,28,112,...,5,3,35,70,87,172,0,16,4,8
1,67d88578,France,48f165d0,93,5,148,13,2,20,119,...,7,1,17,75,93,148,0,18,8,13
2,67d88578,France,4a49c716,124,7,0,0,0,33,157,...,9,2,37,91,124,100,0,18,7,10
3,67d88578,France,4deaa73c,140,7,117,11,3,23,174,...,11,4,45,103,140,117,1,8,4,11
4,67d88578,France,5222ebe1,96,7,0,0,0,33,131,...,9,3,42,62,96,187,2,12,3,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,3a1983b3,Fiji,aa4c09a1,99,3,109,8,6,23,130,...,0,0,0,0,0,0,0,0,0,0
287,3a1983b3,Fiji,b901c3ad,109,6,55,18,4,22,177,...,0,0,0,0,0,0,0,0,0,0
288,3a1983b3,Fiji,c0c4c528,116,6,145,13,8,20,161,...,0,0,0,0,0,0,0,0,0,0
289,3a1983b3,Fiji,cc6f63ba,129,5,91,21,6,24,176,...,0,0,0,0,0,0,0,0,0,0


In [46]:
## Joining data
## Columns of stats not collected using the players dataset
team_stats_copy = team_stats
team_stats = team_stats_copy[['team_id', 'team', 'game_date', 'team_vs', 'team_vs_id', 'match_id', '22m_entries', '22m_conversion',
                        'post_contact_metres', 'ruck_speed_0_3_pct', 'ruck_speed_3_6_pct', 'ruck_speed_6_plus_pct',
                        'possession', 'territory', 'scrums_total', 'mauls_total', 'rucks_total']]

team_stats = team_stats.rename(columns={"team": "team_name"}) # renamed for the purpose of joining

In [78]:
## Join this with data collected from players data set

potential_model_set = pd.merge(team_stats, match_stats, how="left", on = ['team_id', 'team_name', 'match_id'])

potential_model_set

,team_id,team_name,game_date,team_vs,team_vs_id,match_id,22m_entries,22m_conversion,post_contact_metres,ruck_speed_0_3_pct,...,offload,penalty_goals,points,rucks_won,runs,tackles,total_free_kicks_conceded,total_lineouts,turnover_knock_on,turnovers_conceded
0,67d88578,France,20220206,Italy,d0cffe5e,4a49c716,NaN,NaN,NaN,NaN,...,9.0,2.0,37.0,91.0,124.0,100.0,0.0,18.0,7.0,10.0
1,67d88578,France,20220212,Ireland,165b36e5,71ccf2fb,NaN,NaN,NaN,NaN,...,8.0,6.0,30.0,79.0,101.0,119.0,1.0,14.0,7.0,12.0
2,67d88578,France,20220226,Scotland,4409215c,eb863c39,NaN,NaN,NaN,NaN,...,10.0,0.0,36.0,37.0,75.0,153.0,1.0,13.0,6.0,13.0
3,67d88578,France,20220311,Wales,094dae60,6b6585fc,NaN,NaN,NaN,NaN,...,10.0,2.0,13.0,68.0,98.0,131.0,1.0,14.0,9.0,15.0
4,67d88578,France,20220319,England,f60f757d,cc3c0a71,NaN,NaN,NaN,NaN,...,13.0,2.0,25.0,70.0,98.0,130.0,0.0,11.0,8.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524,3a1983b3,Fiji,20241123,Ireland,165b36e5,8923cd55,17.0,3.06,356.0,65.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525,3a1983b3,Fiji,20250830,Tonga,be110f80,b901c3ad,12.0,2.42,327.0,63.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
526,3a1983b3,Fiji,20250906,Samoa,23778ceb,aa4c09a1,8.0,1.88,286.0,61.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
527,3a1983b3,Fiji,20250915,Canada,1039f7c4,55e97718,8.0,0.88,216.0,70.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [79]:
## Home/Away Designation

drop_cols = ['team_name', 'team_vs', 'team_vs_id', 'game_date', 'score', 'team_vs_score']

potential_model_set.drop(columns = [c for c in drop_cols if c in potential_model_set.columns])

stat_cols = [c for c in potential_model_set.columns if c not in ('team_id', 'match_id')]

home_team_stats = potential_model_set.rename(columns={
    'team_id': 'home_team_id',
    **{c: f'home_{c}' for c in stat_cols}
})

away_team_stats = potential_model_set.rename(columns={
    'team_id': 'away_team_id',
    **{c: f'away_{c}' for c in stat_cols}
})



potential_model_set = pd.merge(test_match_results, home_team_stats, on = ['match_id', 'home_team_id'], how = 'left')

potential_model_set = potential_model_set.merge(away_team_stats, on = ['match_id', 'away_team_id'], how = 'left')

In [80]:
## Removing Neutral Matches

potential_model_set = potential_model_set[potential_model_set['neutral_match'] != "neutral"]

potential_model_set = potential_model_set.drop(columns=['neutral_match', 'home_game_date', 'away_game_date'])



In [ ]:
## Eventually will delete later
match_info = potential_model_set[['match_id', 'date', 'season', 'competition_id', 'competition', 
                                'venue_id', 'venue', 'home_team_id', 'home_team', 'away_team_id'
                                ,'away_team', 'home_score', 'away_score', 'winner', 'home_team_win']]


game_stats = potential_model_set.drop(columns = ['season', 'competition_id', 'competition', 'venue_id', 'venue',
                                    'home_score', 'away_score', 'winner', 'home_team_win', 'home_team_vs_id', 'away_team_vs_id',
                                    'home_team_name', 'home_team_vs', 'away_team_name', 'away_team_vs'])



In [ ]:
## Adding potential features to model; will continue to experiment (look at iPad notes for features!!!)

model_set = potential_model_set['match_id', 'date', 'season', 'competition_id', 'competition','venue_id', 'venue', 
                                        'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_score', 'away_score',
                                        'winner', 'home_team_win', 'home_carries']